# Snort Alert Matching

- This notebook matches Snort alerts to the network flows of the CICIDS2017 dataset (Tuesday–Friday) and computes Snort's detection metrics.
- Matching runs day by day over a bidirectional 5-tuple (src IP, src port, dst IP, dst port, protocol) with a time-window check.
- An alert counts as a hit if its timestamp lies exactly within the flow's time window (step 1), or for short flows under 1 second with a +1s tolerance (step 2).

1. Load flows
2. Load alerts
3. Define matching functions
4. Run matching
5. Output results and metrics
6. Save residual
7. Sensitivity analysis: effect of different tolerance windows on recall and FPR
8. Matching validation: spot-check plausibility review of individual alert-flow pairs

In [ ]:
import gc
import json
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap
from pathlib import Path

FLOW_FILE     = Path("/data/flow_csv/prepared/TueWedThuFri-WorkingHours_matching.csv")
ALERT_FILE    = Path("/snort/logs/alert_csvTotal.txt")
RESIDUAL_FILE = Path("/snort/residuals/snort_residuals.csv")
METRICS_FILE  = Path("/snort/residuals/snort_metrics.json")
FIGURES_DIR   = Path("/snort/figures")

# Matching day by day
DAYS      = ["Tuesday", "Wednesday", "Thursday", "Friday"]
TOLERANCE = pd.Timedelta(seconds=1)

## 1. Load Flows

Each row is a network flow with a 5-tuple, timestamps (start/end), and an attack label.  

In [ ]:
flows = pd.read_csv(FLOW_FILE, low_memory=False)
flows.columns        = flows.columns.str.strip()
flows["Label"]          = flows["Label"].str.strip()
flows["Source IP"]      = flows["Source IP"].str.strip()
flows["Destination IP"] = flows["Destination IP"].str.strip()
flows["Timestamp"]      = pd.to_datetime(flows["Timestamp"],     format="%Y-%m-%d %H:%M:%S.%f")
flows["Timestamp End"]  = pd.to_datetime(flows["Timestamp End"], format="%Y-%m-%d %H:%M:%S.%f")
flows["Source Port"]      = pd.to_numeric(flows["Source Port"])
flows["Destination Port"] = pd.to_numeric(flows["Destination Port"])
flows["Protocol"]         = pd.to_numeric(flows["Protocol"])

# Save original columns before adding helper columns
output_columns       = flows.columns.tolist()
flows["Flow Row ID"] = flows.index
flows["Day"]         = flows["Timestamp"].dt.day_name()

print(f"Flows loaded: {len(flows):,}")
print(f"Total columns: {len(output_columns)}")
flows["Label"].value_counts().rename_axis("Label").reset_index(name="Flow Count")

## 2. Load Alerts

- Snort outputs alerts in CSV format: timestamp, protocol, src IP, src port, dst IP, dst port, rule ID, message
- Alert timestamps are corrected by 3 hours, since the PCAP timestamps in CICIDS2017 are shifted relative to the flow timestamps
- Non-IP protocols (eth, ARP) are removed – they don't occur in the flow CSV and therefore can't be matched

In [ ]:
ALERT_COLUMNS = ["Alert_TS", "Alert_Protocol", "Src_IP", "Src_Port",
                 "Dst_IP",   "Dst_Port",  "Rule_ID", "Msg"]

alerts = pd.read_csv(ALERT_FILE, header=None, names=ALERT_COLUMNS,
                     skipinitialspace=True, low_memory=False)

for column in ["Alert_TS", "Alert_Protocol", "Src_IP", "Dst_IP"]:
    alerts[column] = alerts[column].astype(str).str.strip().str.strip('"')

# Timezone correction: Snort timestamps are 3 hours ahead of the flow timestamps
alerts["Alert_TS"] = (
    pd.to_datetime("2017/" + alerts["Alert_TS"], format="%Y/%m/%d-%H:%M:%S.%f")
    - pd.Timedelta(hours=3)
)

# Flow CSV encodes protocols as a number (TCP=6, UDP=17, ICMP=1, IP=0)
# unknown protocols become NaN (eth, ARP – cannot be matched to flows)
protocol_numbers = {"TCP": 6, "UDP": 17, "ICMP": 1, "IP": 0}
alerts["Proto_Num"] = alerts["Alert_Protocol"].str.upper().map(protocol_numbers)

# Remove rows where Proto_Num is empty
alerts = alerts.dropna(subset=["Proto_Num"])

alerts["Src_Port"] = pd.to_numeric(alerts["Src_Port"])
alerts["Dst_Port"] = pd.to_numeric(alerts["Dst_Port"])
alerts["Alert_ID"] = alerts.index
alerts["Day"]      = alerts["Alert_TS"].dt.day_name()

print(f"Alerts loaded: {len(alerts):,}")
alerts["Day"].value_counts().rename_axis("Day").reset_index(name="Alert Count")

## 3. Matching Functions

- merge_bidirectional: joins alerts with flows over the 5-tuple in both directions, since Snort writes some alerts with swapped src/dst addresses
- find_unique_ids: checks the time window and keeps only alerts that match exactly one flow (unique). If an alert matches multiple flows, it is discarded.

In [ ]:
# Only the 9 columns relevant for matching
FLOW_COLUMNS = ["Flow Row ID", "Source IP", "Source Port", "Destination IP",
                "Destination Port", "Protocol", "Timestamp", "Timestamp End", "Label"]

# Joins alerts with flows over the 5-tuple
def merge_bidirectional(day_alerts, day_flows):
    alert_columns   = ["Src_IP", "Src_Port", "Dst_IP", "Dst_Port", "Proto_Num"]
    forward_columns = ["Source IP",      "Source Port",      "Destination IP", "Destination Port", "Protocol"]
    reverse_columns = ["Destination IP", "Destination Port", "Source IP",      "Source Port",      "Protocol"]
    forward = day_alerts.merge(day_flows, left_on=alert_columns, right_on=forward_columns, how="inner")
    reverse = day_alerts.merge(day_flows, left_on=alert_columns, right_on=reverse_columns, how="inner")
    # Merge both directions, remove duplicates
    return (pd.concat([forward, reverse], ignore_index=True)
              .drop_duplicates(subset=["Alert_ID", "Flow Row ID"]))

def find_unique_ids(tuple_pairs, tolerance=pd.Timedelta(0)):
    # Step 1 (tolerance=0): alert must lie exactly within the flow's time window
    # Step 2 (tolerance=1s): window is extended by +1s – for short flows with a truncated timestamp
    hits = tuple_pairs[
        (tuple_pairs["Alert_TS"] >= tuple_pairs["Timestamp"]) &
        (tuple_pairs["Alert_TS"] <= tuple_pairs["Timestamp End"] + tolerance)
    ]
    flows_per_alert = hits.groupby("Alert_ID")["Flow Row ID"].nunique()
    unique_ids      = flows_per_alert[flows_per_alert == 1].index
    return set(hits[hits["Alert_ID"].isin(unique_ids)]["Flow Row ID"])

print("Functions defined.")

## 4. Run Matching

- Step 1 – exact matching: alert timestamp must lie exactly within the flow's time window
- Step 2 – tolerance matching: flows not detected in step 1 and shorter than 1 sec get a +1s buffer on the end of the time window 

In [ ]:
detected_step1 = set()
detected_step2 = set()

for day in DAYS:
    print(f"Matching: {day} ...", end=" ")

    # Load only the flows and alerts of the current day
    day_flows  = flows.loc[flows["Day"] == day, FLOW_COLUMNS].copy()
    day_alerts = alerts.loc[alerts["Day"] == day].copy()

    # Step 1: 5-tuple merge + exact time window – add detected flow IDs to the set
    tuple_pairs = merge_bidirectional(day_alerts, day_flows)
    detected_step1.update(find_unique_ids(tuple_pairs))
    del tuple_pairs

    # Step 2: only flows that step 1 did not detect AND that last less than 1s
    not_in_step1 = day_flows["Flow Row ID"].isin(detected_step1) == False
    short_flows  = day_flows[not_in_step1 & ((day_flows["Timestamp End"] - day_flows["Timestamp"]) < TOLERANCE)].copy()

    if len(short_flows) > 0:
        tuple_pairs_short = merge_bidirectional(day_alerts, short_flows)
        newly_detected = find_unique_ids(tuple_pairs_short, tolerance=TOLERANCE) - detected_step1
        detected_step2.update(newly_detected)
        del tuple_pairs_short

    del day_flows, short_flows, day_alerts
    gc.collect()
    print("done")

# unites both sets – each flow ID appears only once
all_detected = detected_step1 | detected_step2
print(f"\nStep 1 detected flows (exact):           {len(detected_step1):>8,}")
print(f"Step 2 detected flows (+1s tolerance):    {len(detected_step2):>8,}")
print(f"Total detected:                           {len(all_detected):>8,}")

## 5. Results and Metrics


In [ ]:
# Set result columns: True/False per flow for detection, attack type, and which step
flows["Snort Detected"] = flows["Flow Row ID"].isin(all_detected)
flows["Is Attack"]      = flows["Label"].str.upper() != "BENIGN"
flows["Step1"]          = flows["Flow Row ID"].isin(detected_step1)
flows["Step2"]          = flows["Flow Row ID"].isin(detected_step2)

def metrics(df):
    tp  = int((df["Is Attack"] &  df["Snort Detected"]).sum())
    fp  = int((~df["Is Attack"] & df["Snort Detected"]).sum())
    fn  = int((df["Is Attack"] & ~df["Snort Detected"]).sum())
    tn  = int((~df["Is Attack"] & ~df["Snort Detected"]).sum())

    p   = tp / (tp + fp) if tp + fp > 0 else 0      # Precision
    r   = tp / (tp + fn) if tp + fn > 0 else 0      # Recall
    f1  = 2*p*r / (p+r)  if p + r  > 0 else 0       # F1-Score
    fpr = fp / (fp + tn)  if fp + tn > 0 else 0     # FPR
    return {"TP": tp, "FP": fp, "FN": fn, "TN": tn,
            "Precision (%)": round(p*100, 2), "Recall (%)": round(r*100, 2),
            "F1-Score (%)": round(f1*100, 2), "FPR (%)": round(fpr*100, 4)}

print("Overall metrics")
display(pd.DataFrame([metrics(flows)], index=["Snort"]))

print("Metrics per day")
pd.DataFrame({day: metrics(flows[flows["Day"] == day]) for day in DAYS}).T

In [ ]:
# Confusion matrix Snort – TP, FP, FN, TN
m = metrics(flows)
cm = [[m["TN"], m["FP"]],
      [m["FN"], m["TP"]]]

fig, ax = plt.subplots(figsize=(5, 4))
cmap_yellow = LinearSegmentedColormap.from_list("yellow", ["#FFFDF5", "#FDC400"])
im = ax.imshow(cm, cmap=cmap_yellow)

ax.set_xticks([0, 1]); ax.set_xticklabels(["No alert", "Alert"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["Benign", "Attack"])
ax.set_xlabel("Snort prediction")
ax.set_ylabel("Actual")
ax.set_title("Confusion matrix of Snort")

for i in range(2):
    for j in range(2):
        ax.text(j, i, f'{cm[i][j]:,}', ha="center", va="center",
                color="black", fontsize=11)

cb = plt.colorbar(im, ax=ax)
cb.formatter.set_useOffset(False)
cb.formatter.set_scientific(False)
cb.update_ticks()

plt.tight_layout()
plt.savefig(FIGURES_DIR / "confusion_matrix_snort.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Detection performance by attack class
attacks = (flows[flows["Is Attack"]]
           .groupby("Label")
           .agg(Total=("Label", "count"),
                Detected=("Snort Detected", "sum"),
                Step1=("Step1", "sum"),
                Step2=("Step2", "sum")))
attacks["Step1 (%)"] = (attacks["Step1"] / attacks["Total"] * 100).round(2)
attacks["Step2 (%)"] = (attacks["Step2"] / attacks["Total"] * 100).round(2)
attacks["Rate (%)"]  = (attacks["Detected"] / attacks["Total"] * 100).round(2)

print("Detection by attack class")
attacks.sort_values("Total", ascending=False)[["Total", "Detected", "Step1 (%)", "Step2 (%)", "Rate (%)"]]

## 6. Save Residual

- All flows that Snort did not detect are saved as a CSV
- Results for Isolation Forest are saved as JSON

In [ ]:
not_detected = flows["Snort Detected"] == False
residual = flows.loc[not_detected, output_columns].copy()
residual["Timestamp"]     = residual["Timestamp"].dt.strftime("%Y-%m-%d %H:%M:%S.%f")
residual["Timestamp End"] = residual["Timestamp End"].dt.strftime("%Y-%m-%d %H:%M:%S.%f")
residual.to_csv(RESIDUAL_FILE, index=False)

print(f"Flows in the residual: {len(residual):,}")
print(f"\nLabel distribution in the residual:")
display(residual["Label"].value_counts().rename_axis("Label").reset_index(name="Flow Count"))

# Save metrics as JSON – read in by isolationForestFinal.ipynb
overall_metrics = metrics(flows)
with open(METRICS_FILE, "w") as f:
    json.dump(overall_metrics, f, indent=2)


## 7. Sensitivity Analysis – Tolerance Window

- Step 2 of the matching uses a +1s tolerance window for short flows with a truncated timestamp
- The analysis checks how strongly the Snort metrics depend on this choice
- The short-flow threshold stays fixed at < 1s - only the time window is tested: +0.5s, +1.0s, +1.5s and +2.0s

In [ ]:
# Recompute matching for each tolerance value
TEST_WINDOWS_SEC     = [0.5, 1.0, 1.5, 2.0]
SHORT_FLOW_THRESHOLD = pd.Timedelta(seconds=1)

detected_per_tolerance = {}
table1_rows = []

for sec in TEST_WINDOWS_SEC:
    tolerance = pd.Timedelta(seconds=sec)
    detected_s2_test = set()

    for day in DAYS:
        day_flows  = flows.loc[flows["Day"] == day, FLOW_COLUMNS].copy()
        day_alerts = alerts.loc[alerts["Day"] == day].copy()

        # Candidates: flows that S1 did not detect AND that last less than 1s
        not_in_s1   = ~day_flows["Flow Row ID"].isin(detected_step1)
        short_flows = day_flows[
            not_in_s1 & ((day_flows["Timestamp End"] - day_flows["Timestamp"]) < SHORT_FLOW_THRESHOLD)
        ].copy()

        if len(short_flows) > 0:
            tuple_pairs = merge_bidirectional(day_alerts, short_flows)
            # Time window is extended with the current test value
            detected_s2_test.update(
                find_unique_ids(tuple_pairs, tolerance=tolerance) - detected_step1
            )

        del day_flows, short_flows, day_alerts
        gc.collect()

    detected_per_tolerance[sec] = detected_s2_test

    # Compute overall metrics for this tolerance value
    all_test = detected_step1 | detected_s2_test
    detected = flows["Flow Row ID"].isin(all_test)
    tp = int(( flows["Is Attack"] &  detected).sum())
    fp = int((~flows["Is Attack"] &  detected).sum())
    fn = int(( flows["Is Attack"] & ~detected).sum())
    tn = int((~flows["Is Attack"] & ~detected).sum())

    table1_rows.append({
        'Tolerance (s)':     sec,
        'Step-1 Matches':    len(detected_step1),
        'Step-2 Matches':    len(detected_s2_test),
        'TP': tp, 'FP': fp,
        'Recall (%)': round(tp / (tp + fn) * 100, 2),
        'FPR (%)':    round(fp / (fp + tn) * 100, 4),
    })

# Table 1: output recall and FPR for each tolerance value
print("Table 1: Effect of the tolerance window (short-flow threshold fixed: < 1s)")
display(pd.DataFrame(table1_rows).set_index('Tolerance (s)'))

# Table 2: prepare and output the gain per attack class
detected_s1 = flows["Flow Row ID"].isin(detected_step1)
attack_labels = (flows[flows["Is Attack"]]
                 .groupby("Label")["Label"].count()
                 .sort_values(ascending=False).index.tolist())

table2_rows = []
for label in attack_labels:
    mask  = flows["Label"] == label
    total = int(mask.sum())
    row   = {'Attack Class': label, 'Total': total}
    row['S1 only'] = int((mask & detected_s1).sum())
    # Gain: how many flows of this class are additionally detected by S2 on top of S1
    for sec in TEST_WINDOWS_SEC:
        s2_mask = flows["Flow Row ID"].isin(detected_per_tolerance[sec])
        row[f'Gain +{sec}s'] = int((mask & s2_mask).sum())
    table2_rows.append(row)

print("\nTable 2: Gain from step 2 per attack class (absolute)")
display(pd.DataFrame(table2_rows).set_index('Attack Class'))

## 8. Matching Validation – Spot-Check Plausibility Review

- For 10 randomly selected flows each from step 1 and step 2, the corresponding alert is reconstructed
- Checked: alert timestamp lies within the flow's time window, 5-tuple matches (directly or bidirectionally swapped), label and alert message are plausible
- A complete manual validation of all matches is not possible given the data volume

In [ ]:
# Draw a random sample from S1 and S2
import random
random.seed(14)

sample_s1 = set(random.sample(sorted(detected_step1), min(10, len(detected_step1))))
sample_s2 = set(random.sample(sorted(detected_step2), min(10, len(detected_step2))))

# Re-run matching for the sample
parts = []
for step, ids, tol in [(1, sample_s1, pd.Timedelta(0)), (2, sample_s2, TOLERANCE)]:
    sample = flows[flows["Flow Row ID"].isin(ids)][FLOW_COLUMNS]
    hits = merge_bidirectional(alerts, sample)
    hits = hits[
        (hits["Alert_TS"] >= hits["Timestamp"]) &
        (hits["Alert_TS"] <= hits["Timestamp End"] + tol)
    ].drop_duplicates("Flow Row ID").copy()
    hits["S"]         = step
    hits["Alert Src"] = hits["Src_IP"]          + ":" + hits["Src_Port"].astype(int).astype(str)
    hits["Alert Dst"] = hits["Dst_IP"]          + ":" + hits["Dst_Port"].astype(int).astype(str)
    hits["Flow Src"]  = hits["Source IP"]       + ":" + hits["Source Port"].astype(int).astype(str)
    hits["Flow Dst"]  = hits["Destination IP"]  + ":" + hits["Destination Port"].astype(int).astype(str)
    parts.append(hits[["S", "Alert_TS", "Timestamp", "Timestamp End",
                        "Alert Src", "Alert Dst", "Flow Src", "Flow Dst", "Alert_Protocol", "Label", "Msg"]])

# Output result table
df_sample = (pd.concat(parts)
               .rename(columns={"Timestamp": "Flow Start", "Timestamp End": "Flow End", "Alert_Protocol": "Proto"})
               .reset_index(drop=True))
pd.set_option('display.max_colwidth', 80)
print(f"Sample: {len(df_sample)} alert-flow pairs (10 each from S1/S2, seed=42)")
print("S=1: Alert_TS must lie between Flow Start and Flow End")
print("S=2: Alert_TS may lie up to +1s after Flow End (short-flow tolerance)")
display(df_sample)